In [115]:
import pandas as pd
import nibabel as nib
import numpy as np
import yaml
import os
import sys
import argparse
import glob

from sklearn.model_selection import train_test_split

In [116]:
ico_res = 6
ico_down_res = 2
down_num_vertices = 153 
down_num_patches = 320

## destination privacy etc? ask janine

patch_extraction_path = "/ceph/chpc/shared/janine_bijsterbosch_group/lucas_scratch/SiT_structural_encoder/patch_extraction"
data_path = "/ceph/chpc/shared/janine_bijsterbosch_group/lucas_scratch/HCP_YA"

scans_ids = np.array([f for f in os.listdir(f"{data_path}/resampled_icos") if f.isnumeric()])
print(len(scans_ids))

1113


In [117]:
indices_mesh_triangles=pd.read_csv(f'{patch_extraction_path}/triangle_indices_ico_{ico_res}_sub_ico_{ico_down_res}.csv')
id_label_info = pd.read_csv(f"{data_path}/unrestricted_hcp_freesurfer.csv").to_numpy()[:, 0:2]
restricted_access_data = pd.read_csv(f"{data_path}/HCP_YA_subjects_2026_10_22_12_06_42.csv")

In [118]:
family_info = restricted_access_data.iloc[:, [0, 5,6,7]].to_numpy()

In [129]:
def preprocess_and_sort(ids, family_ids, train_split, test_split):
    sex_dict = {str(row[0]): row[1] for row in id_label_info}
    result_dict = {'train':[], 'test':[], 'validation':[]}

    train_family_ids, temp = train_test_split(np.unique(family_ids), train_size=train_split) 
    test_family_ids, val_family_ids = train_test_split(temp, train_size=test_split/(1-train_split))
    
    for i in ids:
        fam_id = family_info[family_info[: , 0] == int(i), 1]
        if fam_id in train_family_ids:
            result_dict['train'].append(int(i))
        if fam_id in test_family_ids:
            result_dict['test'].append(int(i))
        if fam_id in val_family_ids:
            result_dict['validation'].append(int(i))

    print("train size: ", len(result_dict['train']))
    print("test size: ", len(result_dict['test']))
    print("validation size: ", len(result_dict['validation']))

    for split in ['train', 'test', 'validation']:
        
        selected_ids = result_dict[split]
        num_subjects = len(selected_ids)
        data = np.zeros((num_subjects*2, 1, down_num_patches, down_num_vertices)) # 1 == number of channels
        labels = np.zeros((num_subjects*2, 1))

        for i, idx in enumerate(selected_ids):
            subject_sex = int(sex_dict[str(idx)] == "F")
            
            labels[i] = subject_sex
            labels[i+num_subjects] = subject_sex

            L = f"{data_path}/resampled_icos/{idx}/{idx}_ico{ico_res}.L.shape.gii"
            loaded_L_hemi = np.array(nib.load(L).agg_data()) # in the future, w/ more channels we can change this
        
            R = f"{data_path}/resampled_icos/{idx}/{idx}_ico{ico_res}.R.shape.gii"
            loaded_R_hemi = np.array(nib.load(R).agg_data())

            for j in range(down_num_patches):
            
                indices_to_extract = indices_mesh_triangles[str(j)].to_numpy()
                data[i,:,j,:] = loaded_L_hemi[indices_to_extract]
                data[i+num_subjects,:,j,:] = loaded_R_hemi[indices_to_extract]

                # what we see here: store left side at index i, right side at index i + num_subjects
        result_dict[split] = (data, labels)
    
    return result_dict

In [123]:
family_data = np.unique(family_info[:, 1])
data_package = preprocess_and_sort(scans_ids, family_data, 0.7, 0.15)

771
179
163


In [128]:
for i in ['train', 'test', 'validation']:
    data_file = f"{data_path}/{i}_data.npy"
    label_file = f"{data_path}/{i}_labels.npy"

    np.save(data_file, data_package[i][0])
    np.save(label_file, data_package[i][1])
